In [1]:
import gc
import json
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet34_Weights, resnet34

In [2]:
DATASET_ROOT = Path("/content/datasets")
IMAGE_ROOT = DATASET_ROOT / "images"
MASK_ROOT = DATASET_ROOT / "crack-seg-semantic" / "masks"

MANIFEST_PATH = Path(
    "/content/"
    "vision_unit_02_outputs/"
    "block_02/"
    "semantic_mask_manifest.csv"
)

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs/"
    "block_03"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = 416
BATCH_SIZE = 8
NUM_WORKERS = 2
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

USE_AMP = (DEVICE.type == "cuda")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("AMP:", USE_AMP)
print("Output:", OUTPUT_ROOT)

PyTorch: 2.11.0+cpu
Device: cpu
AMP: False
Output: /content/drive/MyDrive/vision_unit_02_outputs/block_03


**Reproducibility**

In [3]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

**Load only train and validation rows**

In [4]:
required_paths = [
    IMAGE_ROOT / "train",
    IMAGE_ROOT / "val",
    MASK_ROOT / "train",
    MASK_ROOT / "val",
    MANIFEST_PATH,
]

for path in required_paths:
    assert path.exists(), (
        f"Missing: {path}"
    )

manifest = pd.read_csv(MANIFEST_PATH)

working_manifest = (
    manifest[manifest["split"].isin(["train", "val"])]
    .copy()
    .reset_index(drop=True)
)

train_frame = (
    working_manifest[working_manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_frame = (
    working_manifest[working_manifest["split"] == "val"]
    .copy()
    .reset_index(drop=True)
)


print("Train:", len(train_frame))
print("Validation:", len(val_frame))
print("Test rows used:",(working_manifest["split"] == "test").sum())


Train: 3717
Validation: 200
Test rows used: 0


**Dataset with synchronized augmentation**

In [5]:
IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406], dtype=np.float32
).reshape(1, 1, 3)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225], dtype=np.float32
).reshape(1, 1, 3)

In [6]:
class CrackSemanticDataset(Dataset):
    def __init__(self, frame, image_root, mask_root, image_size=416, augment=False):
        self.frame = frame.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.mask_root = Path(mask_root)
        self.image_size = image_size
        self.augment = augment
    
    def __len__(self):
        return len(self.frame)
    
    def _augment(self, image, mask):
        if random.random() < 0.50:
            image = np.flip(image, axis=1)
            mask = np.flip(mask, axis=1)
            
        if random.random() < 0.50:
            image = np.flip(image, axis=0)
            mask = np.flip(mask, axis=0)
        
        rotation_k = random.randint(0, 3)
        
        if rotation_k:
            image = np.rot90(image, k=rotation_k, axes=(0, 1))
            mask = np.rot90(mask, k=rotation_k, axes=(0, 1))
        
        contrast = random.uniform(0.90, 1.10)
        brightness = random.uniformA(-0.05, 0.05)
        
        image = np.clip(
            (image - 0.50) * contrast + 0.50 + brightness,
            0.0, 1.0
        )
        return image, mask
    
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        
        split_name = str(row["split"])
        relative_image = Path(str(row["relative_image"]))
        
        image_path = self.image_root / split_name / relative_image
        
        mask_path = self.mask_root / split_name / relative_image.with_suffix(".png")
        
        image = cv2.imread(str(image_path, cv2.IMREAD_COLOR))
        mask = cv2.imread(str(mask_path, cv2.IMREAD_GRAYSCALE))

        if image is None:
            raise FileNotFoundError(image_path)

        if mask is None:
            raise FileNotFoundError(mask_path)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)
        
        target_size = (self.image_size, self.image_size)
        
        if image.shape[:2] != target_size:
            image = cv2.resize(image, target_size, interpolation=cv2.INTER_LINEAR)
            
            mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
        
        if self.augment:
            image, mask = self._augment(image, mask)
        
        image = (image - IMAGENET_MEAN) / IMAGENET_STD
        
        image = np.ascontiguousarray(image.transpose(2, 0, 1))
        mask = np.ascontiguousarray(mask[None, :, :])
        
        return {
            "image": torch.from_numpy(image).float(),
            "mask": torch.from_numpy(mask).float(),
            "relative_image": str(relative_image),
            "crack_ratio": float(row["crack_ratio"]),
        }